<a href="https://colab.research.google.com/github/lannd3217/Interview_RAG/blob/main/RAG_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import userdata
import os

GIT_TOKEN = userdata.get('GITHUB_TOKEN')
GIT_USER = "lannd3217"
GIT_REPO = "Interview_RAG"
GIT_EMAIL = "lanngocd.17@gmail.com"
GIT_NAME = "LAN DINH"

!git config --global user.email {GIT_EMAIL}
!git config --global user.name {GIT_NAME}
remote_url = f"https://{GIT_TOKEN}@github.com/{GIT_USER}/{GIT_REPO}.git"
!git clone {remote_url}

In [ ]:
%cd Interview_RAG

In [ ]:
%%capture
!pip install -q ragas datasets
!pip install -U langchain langchain-community langchain-openai
!pip install -U langchain langchain-community langchain-text-splitters
!pip install -U pymupdf langchain-community
!pip install -U langchain-huggingface sentence-transformers
!pip install -q sentence-transformers faiss-cpu transformers
!pip install langchain langchain-community langchain-chroma langchain-huggingface pymupdf sentence-transformers
!pip install "unstructured[all-docs]"
# !pip install chromadb
!pip install -q "chromadb>=0.5.0"

!pip install rank_bm25

In [ ]:
## LOAD VECTOR STORE AND BUILD RETRIEVER

from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
PERSIST_DIR = "./interview_vector_db"
COLLECTION_NAME = "interview_prep_collection"

embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL_NAME)

vector_store = Chroma(
    persist_directory="./interview_vector_db",
    embedding_function=embeddings,
    collection_name = "interview_prep_collection"
)

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2} # Fetches top-2 most relevant chunks
)

In [ ]:

from transformers import AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
import torch, warnings

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

pipe = pipeline(
    "text-generation",
    model=MODEL_ID,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1,
    max_new_tokens=256,
    do_sample=False,
    temperature=0.0,
)

llm = HuggingFacePipeline(pipeline=pipe)

def combine_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def make_chat_prompt(context: str, question: str) -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "You are an interview preparation assistant. "
                "Answer only using the provided context. "
                "If the answer is not in the context, say \"I don't know.\""
            ),
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {question}",
        },
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

def to_chat_prompt(inputs: dict) -> str:
    return make_chat_prompt(inputs["context"], inputs["question"])

def clean_answer(text: str) -> str:
    # Keep only after the assistant tag if present
    if "<|assistant|>" in text:
        text = text.split("<|assistant|>", 1)[1].strip()
    # Strip trailing special tokens
    text = text.split("</s>")[0].strip()
    # Keep first 2–3 sentences
    sentences = [s.strip() for s in text.split(".") if s.strip()]
    return ". ".join(sentences) + ("." if sentences else "")

chain = (
    {
        "context": retriever | RunnableLambda(combine_docs),
        "question": RunnablePassthrough(),
    }
    | RunnableLambda(to_chat_prompt)   # build TinyLlama chat prompt
    | llm
    | StrOutputParser()
    | RunnableLambda(clean_answer)
)

# Quick test
print(chain.invoke("What is the best way to handle a technical interview?"))

In [ ]:
eval_items = [
    # ========= ML THEORY & METRICS =========
    {
        "question": "What is the bias-variance tradeoff?",
        "ground_truths": [
            "Bias is error from overly simple assumptions that cause underfitting, while variance is error from overly complex models that overfit; you must trade off bias and variance to minimize total error."
        ],
    },
    {
        "question": "Explain the difference between supervised and unsupervised learning.",
        "ground_truths": [
            "Supervised learning trains on labeled data to map inputs to outputs, while unsupervised learning finds structure or clusters in unlabeled data."
        ],
    },
    {
        "question": "How is k-nearest neighbors different from k-means clustering?",
        "ground_truths": [
            "K-nearest neighbors is a supervised classification algorithm that needs labeled examples, while k-means is an unsupervised clustering algorithm that groups unlabeled points by proximity."
        ],
    },
    {
        "question": "Define precision and recall in a classification problem.",
        "ground_truths": [
            "Recall is the proportion of true positives among all actual positives, while precision is the proportion of true positives among all predicted positives."
        ],
    },
    {
        "question": "What does the ROC curve represent?",
        "ground_truths": [
            "A ROC curve plots the true positive rate against the false positive rate at different decision thresholds and visualizes the tradeoff between sensitivity and false alarms."
        ],
    },
    {
        "question": "What is the F1 score and when is it useful?",
        "ground_truths": [
            "The F1 score is the harmonic mean of precision and recall and is useful when you care about both and when true negatives are less important."
        ],
    },
    {
        "question": "Explain the difference between Type I and Type II errors.",
        "ground_truths": [
            "A Type I error is a false positive where you claim an effect that is not real, while a Type II error is a false negative where you miss a real effect."
        ],
    },
    {
        "question": "What is the difference between L1 and L2 regularization?",
        "ground_truths": [
            "L1 regularization encourages sparsity by driving some weights to exactly zero, while L2 regularization spreads penalty across all weights and keeps many small nonzero values."
        ],
    },
    {
        "question": "How would you handle an imbalanced classification dataset?",
        "ground_truths": [
            "You can handle imbalance by collecting more minority-class data, resampling the dataset, trying algorithms that handle imbalance well, or using metrics like precision, recall, and F1 instead of accuracy."
        ],
    },
    {
        "question": "When would you use classification instead of regression?",
        "ground_truths": [
            "You use classification when the target is a discrete category, such as whether a user will churn, and regression when you need a continuous numeric prediction."
        ],
    },
    {
        "question": "What is the bias-variance decomposition telling us about model error?",
        "ground_truths": [
            "The bias-variance decomposition shows total error as the sum of bias, variance, and irreducible noise, making clear that reducing one component can increase the other."
        ],
    },
    {
        "question": "What is the difference between a generative and a discriminative model?",
        "ground_truths": [
            "A generative model learns the joint distribution of features and labels to generate or model data, while a discriminative model learns the decision boundary and directly models the conditional probability of labels given features."
        ],
    },

    # ========= PRACTICAL INTERVIEW PREP =========
    {
        "question": "How should you prepare for a recruiter screening call for a machine learning role?",
        "ground_truths": [
            "You should connect your experience explicitly to the job description, expand acronyms, and focus on showing that your background is relevant and you can learn quickly and fit the team."
        ],
    },
    {
        "question": "What is an effective strategy for choosing what to study before a technical interview?",
        "ground_truths": [
            "A practical strategy is to identify likely question types from the recruiter, skim notes to find weak areas, and prioritize breadth on topics you are weaker on over re-studying what you already know well."
        ],
    },
    {
        "question": "How can you structure your preparation for coding interviews that include data and ML questions?",
        "ground_truths": [
            "You can mix practice on NumPy and pandas exercises with brainteaser-style problems, focusing on common patterns like array manipulation, sliding windows, and two-pointers instead of brute-forcing every problem."
        ],
    },
    {
        "question": "During a recruiter call, how should you talk about your past projects?",
        "ground_truths": [
            "You should describe projects using language that mirrors the job description, explain algorithms and tools in plain terms, and highlight how your work maps to the required skills."
        ],
    },
    {
        "question": "What is the main goal of behavioral interviews in data science or ML roles?",
        "ground_truths": [
            "Behavioral interviews probe how you handle difficult situations, work with others, and communicate, using past experiences to predict future performance and team fit."
        ],
    },
    {
        "question": "How should a new grad prioritize their time when preparing for multiple interview rounds?",
        "ground_truths": [
            "A new grad should focus on the core skills most likely to be tested—coding, statistics, and ML fundamentals—while also allocating time for behavioral stories and role-specific preparation."
        ],
    },
    {
        "question": "What is a sensible interview prep plan for Meta-style ML interviews?",
        "ground_truths": [
            "A Meta-style plan includes brushing up on probability and statistics, practicing ML theory questions, and doing coding practice in Python for data structures and data manipulation, aligned with Meta’s prep guides."
        ],
    },
    {
        "question": "How can reviewing past mistakes help you improve between interview rounds?",
        "ground_truths": [
            "Reviewing questions you fumbled helps you identify weak topics, adjust your study plan to target those gaps, and avoid repeating the same mistakes in later rounds."
        ],
    },
    {
        "question": "Why is it important to think out loud during technical interviews?",
        "ground_truths": [
            "Thinking out loud lets interviewers follow your reasoning, correct wrong assumptions, and see how you approach problems, even if your code or math is not perfect."
        ],
    },
    {
        "question": "How can you avoid over-preparing only for brainteaser questions when targeting data science roles?",
        "ground_truths": [
            "You should balance brainteaser coding practice with data-focused exercises, SQL and pandas practice, ML theory, and case studies that reflect the actual work of data roles."
        ],
    },

    # ========= CAREER / NEW-GRAD & MINDSET (23–26) =========
    {
        "question": "Why do referrals matter in the machine learning and data science job search?",
        "ground_truths": [
            "Referrals can move your resume to the top of the stack or skip early screening, increasing the effectiveness of each application compared with cold applying."
        ],
    },
    {
        "question": "What mindset should a new-grad data scientist have when being asked to build agentic AI without prior experience?",
        "ground_truths": [
            "They should treat it as a chance to get paid to learn, clarify expectations with stakeholders, and focus on iterating from small proofs of concept instead of expecting to be fully expert on day one."
        ],
    },
    {
        "question": "What is a realistic expectation about on-the-job learning for new data scientists?",
        "ground_truths": [
            "New data scientists should expect to continuously upskill on new tools and methods, often learning on the job as technology and company priorities evolve."
        ],
    },
    {
        "question": "Why is networking emphasized as a key part of breaking into ML roles?",
        "ground_truths": [
            "Networking creates warm connections that can lead to referrals, inside information about roles, and conversations that increase your effectiveness per application over time."
        ],
    },

    # ========= A/B TESTING & STATISTICS (27–30) =========
    {
        "question": "What is the purpose of an A/B test in a product context?",
        "ground_truths": [
            "An A/B test randomly assigns users to control and treatment versions to estimate the causal effect of a change on key metrics such as conversion or click-through rate."
        ],
    },
    {
        "question": "Why is randomization important in A/B testing?",
        "ground_truths": [
            "Randomization helps balance confounders between groups so that differences in outcomes can be attributed to the experiment rather than pre-existing differences."
        ],
    },
    {
        "question": "How would you explain the difference between statistical significance and practical significance in an experiment?",
        "ground_truths": [
            "Statistical significance means an effect is unlikely due to chance under the model, while practical significance asks whether the effect size is large enough to matter for the business."
        ],
    },
    {
        "question": "What cross-validation strategy is appropriate for time series modeling?",
        "ground_truths": [
            "For time series you should use forward-chaining or rolling-origin evaluation where you train on past data and test on future data, preserving chronological order instead of random k-fold splits."
        ],
    },
]


In [ ]:
def run_rag_for_eval_item(item):
    q = item["question"]
    refs = item["ground_truths"]

    # Get contexts separately for RAGAS
    docs = retriever.invoke(q)
    contexts = [doc.page_content for doc in docs]

    # answer = chain.invoke(q)
    answer = chain.invoke(q)

    return {
        "question": q,
        "answer": answer[:800],          # truncate to avoid token limit errors
        "contexts": contexts,
        "reference": refs[0] if isinstance(refs, list) else refs,
    }


ragas_rows = [run_rag_for_eval_item(item) for item in eval_items]


In [ ]:
!pip install -q ragas datasets openai

In [ ]:

import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPEN_AI")
# os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY.")

from datasets import Dataset




In [ ]:
# !pip install -q langchain-google-genai


In [ ]:
hf_dataset = Dataset.from_list(ragas_rows)
hf_dataset
def flatten_reference(example):
    ref = example["reference"]
    example["reference"] = ref[0] if isinstance(ref, list) else ref
    return example

hf_dataset = hf_dataset.map(flatten_reference)
print(hf_dataset[0]["reference"])


In [ ]:
MAX_ANSWER_CHARS = 800  # tune if needed

def truncate_answer(example):
    example["answer"] = example["answer"][:MAX_ANSWER_CHARS]
    return example

hf_dataset = hf_dataset.map(truncate_answer)


In [ ]:
from ragas import evaluate

from ragas.metrics import Faithfulness, ContextPrecision

metrics = [Faithfulness(), ContextPrecision()]

# subset = hf_dataset.select(range(min(10, len(hf_dataset))))

results = evaluate(
    dataset=hf_dataset,
    metrics=metrics,
#     run_config=RunConfig(max_tokens=4096),
)

df = results.to_pandas()
df.head()


In [ ]:
df.to_csv("ragas_results.csv", index=False)

In [ ]:
import json

with open("ragas_eval_dataset.json", "w") as f:
    json.dump(ragas_rows, f, indent=2)


In [ ]:

print(df[["user_input", "faithfulness", "context_precision"]].to_string())
print("\nMean scores:")
print(df[["faithfulness", "context_precision"]].mean())


In [ ]:
print("=== Worst by Faithfulness ===")
print(df.sort_values("faithfulness").head(3)[["user_input", "faithfulness", "context_precision"]])

In [ ]:
!git add --all

In [ ]:
!git commit -a -m "Updated eval"

In [ ]:
!git push

In [ ]:

def tag_question_type_by_index(idx: int) -> str:
    # 0-based indexing
    if idx <= 11:
        return "ml_concepts"
    if 12 <= idx <= 21:
        return "interview_prep"
    if 22 <= idx <= 25:
        return "career_mindset"
    if 26 <= idx <= 29:
        return "ab_testing_stats"
    return "other"
df = df.reset_index(drop=True)
df["question_type"] = df.index.map(tag_question_type_by_index)
df["question_type"].value_counts()


In [ ]:
type_stats = (
    df.groupby("question_type")[["faithfulness", "context_precision"]]
      .mean()
      .sort_values("faithfulness", ascending=False)
)

print(type_stats.round(3))
